In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
import sys
sys.path.insert (0,"../part1")

In [ ]:
import part1

In [ ]:
dataset, warnings=part1.load_dataset("rowdata.csv")

In [ ]:
print(warnings)

In [ ]:
#GDP-Unemployment Analysis class can be used by member 4: comparasion, full available sample with no period selection
class GDPUnemploymentAnalysis:
    def __init__(self,dataset):
        self.dataset=dataset
        
    def run_correlation(self, country,unemployment_code, start_year=None, end_year=None): 
        
    #years are none because we want to maximize the sample size
        
    #pearson correlation analysis
    
        years, x_data, y_data=self.dataset.get_regression_data(country,unemployment_code=unemployment_code)
    
        corr, p_value=stats.pearsonr(x_data,y_data)
    
        return {
            "type": "correlation",
            "country": country,
            "unemployment_code": unemployment_code,
            "correlation": float(corr),
            "p_value":float(p_value), 
            "n_observations":len (years)
        }

In [ ]:
analysis= GDPUnemploymentAnalysis(dataset)

In [ ]:
#function for printing the results
def display_correlation (country, unemployment_code):
    result_corr=analysis.run_correlation(country, unemployment_code)
    if unemployment_code==part1.YOUTH_CODE:
        unemployment_type="Youth Unemployment"
    else:
        unemployment_type="Total Unemployment"
    print ("CORRELATION RESULTS")  #MAKE BOLD LATER
    print(f"Country: {country}")
    print(f"Unemployment Type: {unemployment_type}") #may be we can change unemployment codes as youth_unemp and total_unemp
    print(f"Number of observations: {result_corr['n_observations']}") 
    print (f"In {country} correlation between GDP growth and annual change in {unemployment_type} is: {result_corr['correlation']:.3f}")
    print (f"P-value: {result_corr['p_value']:.4f}")
    if result_corr ['p_value']<0.05:
        print ("Statistically significant at the 5% level")
    else:
        print ("Not statistically significant at the 5% level")

In [ ]:
#regression analysis
def run_regression(self, country, unemployment_code,start_year=None, end_year=None):
    years, x_data, y_data=self.dataset.get_regression_data(country, unemployment_code=unemployment_code)
    reg=stats.linregress(x_data,y_data)
    return {
        "type": "regression",
        "country": country,
        "unemployment_code": unemployment_code,
        "slope":float(reg.slope),
        "intercept":float(reg.intercept),
        "r_squared":float(reg.rvalue**2),
        "p_value":float (reg.pvalue),
        "std_error":float (reg.stderr),
        "n_observations": len (years) 
    }

In [ ]:
#print regression results
def display_regression (country, unemployment_code):
    result_reg=run_regression(analysis, country, unemployment_code)
    if unemployment_code==part1.YOUTH_CODE:
        unemployment_type="Youth Unemployment"
    else:
        unemployment_type="Total Unemployment"
    print ("LINEAR REGRESSION RESULTS")  #MAKE BOLD LATER
    print(f"Country: {country}")
    print(f"Unemployment Type: {unemployment_type}") #may be we can change unemployment codes as youth_unemp and total_unemp
    print(f"Number of observations: {result_reg['n_observations']}") 
    print(f"Slope: {result_reg['slope']:.4f}") 
    print(f"R_Squared: {result_reg['r_squared']:.4f}") 
    print (f"P-value: {result_reg['p_value']:.4f}")
 #Statistical significance
    if result_reg['p_value']<0.05:
        print ("Statistically significant at the 5% level")
    else:
        print ("Not statistically significant at the 5% level")
    #regression equation for better understanding the model
    print("Estimated Regression Equation:") 
    print(f"Unemployment Change= {result_reg['intercept']:.3f}"f"+({result_reg['slope']:.3f})*GDP Growth")


    #interpretation of the results
    if result_reg['slope']<0:
        direction="decrease"
    elif result_reg['slope']>0:
        direction="increase"
    else:
        direction=None
        
    if direction is None:
    
         print ("The estimated slope is zero")
    else:
         print (f"In {country} 1-percentage-point increase in GDP growth is associated with a {abs(result_reg['slope']):.3f}-percentage-point "
               f"{direction} in the annual change in the {unemployment_type.lower()} rate.")

In [ ]:
#test print
display_regression("Spain", part1.UNEMPLOYMENT_CODE)  #this code only for checks, delete later


In [ ]:
#regression plot
def plot_regression (country, unemployment_code, start_year=None, end_year=None):
    result_reg=run_regression(analysis, country, unemployment_code, start_year, end_year)
    years, x_data, y_data=analysis.dataset.get_regression_data(country, unemployment_code=unemployment_code)
    x_sorted=np.sort(x_data)
    reg_line=(result_reg["intercept"]+result_reg["slope"]*x_sorted)
    
    if unemployment_code==part1.YOUTH_CODE:
        unemployment_type="Youth Unemployment"
    else:
        unemployment_type="Total Unemployment"
    plt.figure(figsize=(15,7.5))
    plt.scatter(x_data, y_data, label="Actual Data", c="grey")
    plt.plot(x_sorted, reg_line, label="Fitted Regression Line", c="red")
    plt.xlabel("GDP Growth (%)")
    plt.ylabel("Annual Unemployment Change (percentage points)")
    plt.title(f"{country}: GDP Growth vs {unemployment_type} Change")

    plt.legend(fontsize=12)
    plt.grid (True)
    plt.show ()

In [ ]:
#test for plot
plot_regression("Germany", part1.UNEMPLOYMENT_CODE)

In [ ]:
#diagnostic tests for regression
def run_diagnostics(self, country, unemployment_code, start_year=None, end_year=None):
    years, x_data, y_data=self.dataset.get_regression_data(country,start_year, end_year, unemployment_code=unemployment_code)
    #use previously defined regression function 
    result_reg=run_regression(self, country, unemployment_code)
    #predicted values and residuals
    y_pred=result_reg["intercept"]+result_reg["slope"]*x_data
    residuals=y_data-y_pred
    #Shapiro-Wilk normality test
    shapiro_stat, shapiro_p=stats.shapiro(residuals)
    return {
        "country": country,
        "unemployment_code": unemployment_code,
        "y_pred": y_pred,
        "residuals": residuals,
        "shapiro_stat": float(shapiro_stat),
        "shapiro_p_value": float(shapiro_p),
        "n_observations": len(years),
        "years": years
    }

In [ ]:
#displaying the Shapiro_Wilk Test results
def display_diagnostics (country, unemployment_code, start_year=None, end_year=None):
    result_diag=run_diagnostics(analysis, country, unemployment_code)
    if unemployment_code==part1.YOUTH_CODE:
        unemployment_type="Youth Unemployment"
    else:
        unemployment_type="Total Unemployment"
    print ("REGRESSION DIAGNOSTICS-RESIDUALS NORMALITY TEST")
    print (f"Country: {country}")
    print (f"Unemployment Type: {unemployment_type}")
    print (f"Shapiro-Wilk Statistics: {result_diag['shapiro_stat']:.4f} ")
    print (f"P-value: {result_diag['shapiro_p_value']:.4f} ")
    if result_diag["shapiro_p_value"]<0.05:
        print ("Evidence against residual normality.")
    else:
        print ("No significant evidence against residual normality.")

In [ ]:
#test for diagnostics
display_diagnostics("United Kingdom", part1.UNEMPLOYMENT_CODE)

In [ ]:
#regression diagnostic plots
def plot_diagnostics (country, unemployment_code,start_year=None, end_year=None):
    result_diag=run_diagnostics(analysis, country, unemployment_code, start_year, end_year)
    years= result_diag["years"]
    y_pred= result_diag["y_pred"]
    residuals=result_diag["residuals"]
    if unemployment_code==part1.YOUTH_CODE:
        unemployment_type="Youth Unemployment"
    else:
        unemployment_type="Total Unemployment"
        
    #PLOT:Residuals vs Fitted Values
    plt.figure(figsize=(15,7.5))
    plt.scatter(y_pred, residuals)
    plt.axhline(y=0, color='red', linestyle="--")
    plt.xlabel("Predicted Unemployment Change (percentage points)")
    plt.ylabel("Residuals (percentage points)")
    plt.title(f"{country}: {unemployment_type}-Residuals vs Fitted Plot")
    plt.grid (True)
    plt.show ()
    
    #Plot2: Residuals over Time
    plt.figure(figsize=(15,7.5))
    plt.plot(years,residuals, marker="o")
    plt.axhline(y=0, color='red', linestyle="--")
    plt.xlabel("Year")
    plt.ylabel("Residuals (percentage points))")
    plt.title(f"{country}: {unemployment_type}-Residuals over Time")
    plt.grid (True)
    plt.show ()

In [ ]:
#test for plots
plot_diagnostics ("Germany", part1.UNEMPLOYMENT_CODE)